# GDL - Midterm n.1
**Student ID: 583309** (Your student "matricola" goes here)

**Submission ID: 9** (The ID from the sheet circulated in classroom goes here)

In the first midterm assignment, you are required to implement a Gaussian Mixture Model (GMM) and the Expectation-Maximization (EM) algorithm. You are allowed to use `numpy` and other non-machine learning libraries, e.g., `pandas`, `matplotlib`.

**Assumptions.** To ease the implementation, we assume, for each Gaussian distribution in the mixture $\mathcal{N}(\mu_k, \Sigma_k)$, that the *covariance matrix is diagonal*. Furthermore, keep in mind that a good implementation of the EM algorithm provides monotonically increasing log-likelihood, *but* can get stuck in local minima; initialization strategies are fundamental (random? sample points? k-means++?).

**Hyperparameter $k$.** When using a GMM, you should ask yourself: *how many mixtures will be in the data?* You can *maximize* the Bayesian Information Criterion (BIC) to select the number of categories $k$ on your training set. Formally,
$$
\text{BIC}
=
\log P(X\mid \theta) - \frac{|\theta|}{2}\log n,
$$
where $n$ is the number of samples in the training set, $\theta=\{\pi,\mu,\sigma\}$ the parameters of the GMM and $|\theta|$ the number of parameters, i.e., the sum of the number of parameters in $\pi$, $\mu$, and $\sigma$ (*hint*: it also depends on the dimensionality of data $d$!).

**Summary.** Overall, you are required to:
1.  **Implement the GMM class**. Fill the `log_likelihood(samples)`.
2. **Implement the EM algorithm**. Fill the `fit(samples)` method to fit the parameters of a GMM.
3. **Implement the BIC score**. Fill the `bic(samples)` method to perform model selection.
4. **Run training and evaluation**. Select the best scoring model using BIC (Bayesian Information Criterion) for values of $k=\{1,\ldots,6\}$.

**Evaluation.** Your solution will be tested against an hidden test set. For your learning experience, we require you to refrain from using LLM generated code. Violations will be flagged and invalidate the midterm. **Do not alter Sections 3 and 4 of the notebook.**

### 1. Libraries

In [6]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)


print("Libraries imported successfully.")

Libraries imported successfully.


### 2. Gaussian Mixture Model (GMM)

Feel free to play around the implementation. **For evaluation purposes**, the class **must** expose the following attributes and methods:
- `_pi: np.ndarray`
- `_mu: np.ndarray`
- `_sigma: np.ndarray`
- `fit(samples: np.ndarray)`
- `bic(samples: np.ndarray) -> np.ndarray`
- `log_likelihood(samples: np.ndarray) -> np.ndarray`

In [ ]:
def gaussian_pdf(mu: np.ndarray, sigma: np.ndarray, X: np.ndarray) -> np.ndarray:
    diff = X - mu
    exponent = -(diff**2) / (2 * sigma**2)
    pdf = np.exp(exponent) / (np.sqrt(2 * np.pi) * sigma)

    return pdf


class GaussianMixtureModel:
    def __init__(self, n_categories: int, n_features: int):
        self.n_categories = n_categories
        self.n_features = n_features

        # every cluster equiprobable
        self._pi = np.ones(shape=n_categories) * (1 / n_categories)

        # means all at zero
        self._mu = np.random.randn(self.n_categories, self.n_features)

        # variances all at one to prevent initial division by zero
        self._sigma = np.ones((self.n_categories, self.n_features))

    def log_likelihood(self, samples) -> np.ndarray:
        """Computes logP(X|θ)"""

        probabilities = np.zeros(shape=(self.n_categories, samples.shape[0]))

        for m in range(self.n_categories):
            pdf = gaussian_pdf(self._mu[m], self._sigma[m], samples)
            probabilities[m] = np.prod(pdf, axis=1)

        mixture = np.sum(self._pi[:, None] * probabilities, axis=0)

        return np.sum(np.log(mixture))

    def fit(self, samples: np.ndarray, n_iter=100):

        N_samples, D = samples.shape

        prev_log_likelihood = self.log_likelihood(samples)

        for i in range(n_iter):
            # E STEP
            probabilities = np.zeros((self.n_categories, N_samples))

            for m in range(self.n_categories):
                pdf = gaussian_pdf(self._mu[m], self._sigma[m], samples)
                probabilities[m] = np.prod(pdf, axis=1) * self._pi[m]

            denom = np.sum(probabilities, axis=0)
            r = probabilities / denom

            # M STEP
            Nk = np.sum(r, axis=1)

            for m in range(self.n_categories):
                weights = r[m][:, None]

                self._pi[m] = Nk[m] / N_samples

                self._mu[m] = np.sum(weights * samples, axis=0) / Nk[m]

                diff = samples - self._mu[m]

                self._sigma[m] = np.sqrt(np.sum(weights * diff**2, axis=0) / Nk[m])

            print(f"log likelihood: {self.log_likelihood(samples):.4f}")
            curr_log_likelihood = self.log_likelihood(samples)
            if abs(curr_log_likelihood - prev_log_likelihood) < 1e-4:
                print(f"converged in {i} iterations")
                return

            prev_log_likelihood = curr_log_likelihood

    def bic(self, samples: np.ndarray) -> float:
        N, D = samples.shape
        K = self.n_categories

        log_likelihood = self.log_likelihood(samples)

        # numero di parametri
        p = 2 * K * D + (K - 1)

        return log_likelihood - 0.5 * p * np.log(N)


### 3. Training

Trains the model for increasing number of categories and selects the best scoring one in terms of BIC score.

In [8]:
# load training set
train_df = pd.read_csv("train.csv")
print(f"train.csv loaded successfully. n={train_df.shape[0]}, d={train_df.shape[1]}")

# model selection using bayesian information score
seed_everything(9951)
candidate_categories = range(1, 7)
max_bic, best_k, best_gmm = -np.inf, None, None
for k in candidate_categories:
    gmm = GaussianMixtureModel(n_categories=k, n_features=train_df.shape[1])
    gmm.fit(train_df.values)
    ll = gmm.log_likelihood(train_df.values)
    bic_score = gmm.bic(train_df.values)
    print(f"k={k}\tBIC={bic_score:.4f}\tlogP(X|θ)={ll:.4f}")
    if bic_score > max_bic:
        max_bic = bic_score
        best_k = k
        best_gmm = gmm

# print training info
print()
print("Best model")
print(f"k:\t\t{best_k}")
print(f"BIC:\t\t{max_bic:.4f}")
print(f"logP(X|θ):\t{best_gmm.log_likelihood(train_df.values):.4f}")

train.csv loaded successfully. n=800, d=5
log likelihood: -8286.7541
log likelihood: -8286.7541
converged in 1 iterations
k=1	BIC=-8320.1771	logP(X|θ)=-8286.7541
log likelihood: -8283.4373
log likelihood: -8206.9852
log likelihood: -8142.4101
log likelihood: -8099.4865
log likelihood: -8069.4528
log likelihood: -8041.6129
log likelihood: -8018.0852
log likelihood: -8001.5299
log likelihood: -7991.6216
log likelihood: -7986.4247
log likelihood: -7983.9375
log likelihood: -7982.8006
log likelihood: -7982.2862
log likelihood: -7982.0523
log likelihood: -7981.9452
log likelihood: -7981.8958
log likelihood: -7981.8730
log likelihood: -7981.8624
log likelihood: -7981.8575
log likelihood: -7981.8552
log likelihood: -7981.8541
log likelihood: -7981.8536
log likelihood: -7981.8534
log likelihood: -7981.8533
log likelihood: -7981.8532
converged in 24 iterations
k=2	BIC=-8052.0416	logP(X|θ)=-7981.8532
log likelihood: -8066.6203
log likelihood: -8023.8078
log likelihood: -8003.2981
log likelihood:

### 4. Evaluation

To check if you did not break the API, you can use the training file to run the tests.

This is not meaningful for the evaluation of your model, as the true test set is hidden.

In [9]:
# If test.csv does not exist copy train into test
!test -f test.csv || cp train.csv test.csv

In [10]:
# load hidden test set ☠️
test_df = pd.read_csv("test.csv")
print(f"test.csv loaded successfully. n={test_df.shape[0]}, d={test_df.shape[1]}")

# print log-likelihood
test_log_likelihood = best_gmm.log_likelihood(test_df.values)
print(f"Log-likelihood of test data: {test_log_likelihood:.4f}")

# print parameters
print(f"k: {best_gmm.n_categories}")
print()
for cat_id in range(best_gmm.n_categories):
    print(f"π[{cat_id}]:", best_gmm._pi[cat_id])
    print(f"μ[{cat_id}]:", best_gmm._mu[cat_id])
    print(f"σ[{cat_id}]:", best_gmm._sigma[cat_id])
    print()

test.csv loaded successfully. n=800, d=5
Log-likelihood of test data: -7726.1497
k: 4

π[0]: 0.4351782180111222
μ[0]: [-2.01552159  0.56093318 -2.33861819 -0.5066904  -0.10954204]
σ[0]: [1.60105657 1.49968968 1.10956779 1.28400074 1.66985483]

π[1]: 0.1207839916188778
μ[1]: [-2.94815584 -1.75680878  0.81464671  2.46360534 -0.0999458 ]
σ[1]: [1.52676531 1.60639954 0.87368076 1.53954239 0.81371308]

π[2]: 0.12070088806747481
μ[2]: [ 1.87924699  2.39716541 -0.1970332   0.26451986 -0.80131881]
σ[2]: [0.70180329 1.32864147 1.17939247 1.24610842 1.80192215]

π[3]: 0.3233369023025252
μ[3]: [-2.76492954 -0.57485712 -0.45677177  2.05082467 -3.81497619]
σ[3]: [1.25093455 1.77695757 1.02567879 1.48518722 1.24862959]

